In [8]:
from collections import Counter

from datasets import load_dataset

from data import DATASET_NAME, count_classes, prepare_data

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from prompts import MODEL_NAME, SYSTEM_PROMPT, build_messages, format_prompt
from inference import predict


### Загрузка и выборки


In [9]:
dataset = load_dataset(DATASET_NAME)
train_data, few_shot_data, test_data = prepare_data(dataset)

print(f"train: {len(train_data)}, few-shot: {len(few_shot_data)}, test: {len(test_data)}")


train: 5000, few-shot: 5, test: 1000


### Проверка состава



In [10]:
for example in few_shot_data:
    print(f"Оценка: {example['stars']}\n{example['text']}\n")


Оценка: 4
Ставлю за концерт 4 звёзды(((\n   Голоса шикарные, поют волшебно, единственное, что художественный руководитель, Захарченко (ему почёт и уважение), но он забрал на лекции очень много времени ((( Конечно все что он говорил правильно, но на концерт шли отдохнуть и насладиться творчеством! \nДа, и ещё! Из огромного репертуара выбрали не самые лучшие песни для первого части программы. На протяжении часа были колыбельная за колыбельной, чуть не уснули. Вообще, немного было разочарование ((( Могу сравнить Кубанский казачий хор  с бриллиантом, но без огранки.

Оценка: 3
Была именно в этом заведении, купили пончики, картошку, тирамису и десерт (киви).\nВзяли айс-кофе.\nПо бонусным программам нет вопросов, НО есть в Санкт-Петербурге аналогичные булочные и это, конечно, небо и земля.\nОсобенно хочу отметить пироженое "картошка". Честно говоря, это самая невкусная картошка в жизни.\nВ СПб это пироженное можно есть сутками, а тут откусила и выбросила остаток.\nПончики хороши. Кофе - горч

### Загрузка Qwen3-0.6B

При первом запуске скачиваются токенизатор и веса модели. Они остаются в кэше Hugging Face.

In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype="auto")
model = model.to(device)
model.eval()

print(f"Модель: {MODEL_NAME}")
print(f"Устройство: {device}")
print(f"Параметров: {model.num_parameters():,}")


Loading weights: 100%|██████████| 311/311 [00:00<00:00, 545.89it/s]


Модель: Qwen/Qwen3-0.6B
Устройство: cuda
Параметров: 596,049,920


### Промпты zero-shot и few-shot


In [12]:
review = test_data[0]["text"]
zero_shot_messages = build_messages(review)
few_shot_messages  = build_messages(review, few_shot_data)

zero_shot_prompt = format_prompt(tokenizer, zero_shot_messages)
few_shot_prompt = format_prompt(tokenizer, few_shot_messages)

print("ZERO-SHOT:\n")
print(zero_shot_prompt)
print("\nFEW-SHOT:\n")
print(few_shot_prompt)


ZERO-SHOT:

<|im_start|>system
Определи оценку, которую автор поставил организации, по тексту отзыва по шкале от 1 до 5. Где 1 - плохо, а 5 - хорошо. Ответь одной цифрой от 1 до 5<|im_end|>
<|im_start|>user
Делала маникюр и педикюр в этом салоне. Салон неряшливый, видно что работникам всё равно в каких условиях принимать клиентов, да и самим работать. Но в целом услуга оказана. Но такая оценка именно мастеру. Больше к ней не пойду. Во-первых она не надела перчатки для проведения маникюра. Во вторых во время процедуры смотрела телевизор и издергала мне руки. Педикюр сделала хорошо, и в перчатках. <|im_end|>
<|im_start|>assistant
<think>

</think>



FEW-SHOT:

<|im_start|>system
Определи оценку, которую автор поставил организации, по тексту отзыва по шкале от 1 до 5. Где 1 - плохо, а 5 - хорошо. Ответь одной цифрой от 1 до 5<|im_end|>
<|im_start|>user
Ставлю за концерт 4 звёзды(((\n   Голоса шикарные, поют волшебно, единственное, что художественный руководитель, Захарченко (ему почёт и 

### Генерация и разбор ответа

Проверяем два режима на одном отзыве. Жадная генерация: `do_sample=False`, максимум три новых токена. Если оценки 1–5 нет, предсказание — `None`, ответ учитывается в `invalid`.

In [13]:
print("Отзыв:", review)
print("Правильная оценка:", test_data[0]["stars"])

results = {}
for mode, prompt in [("zero_shot", zero_shot_prompt), ("few_shot", few_shot_prompt)]:
    raw_answer, prediction = predict(model, tokenizer, prompt)
    invalid = int(prediction is None)
    results[mode] = {
        "raw_answer": raw_answer,
        "prediction": prediction,
        "invalid": invalid,
    }
    print(f"{mode}: ответ={raw_answer!r}, оценка={prediction}, invalid={invalid}")


Отзыв: Делала маникюр и педикюр в этом салоне. Салон неряшливый, видно что работникам всё равно в каких условиях принимать клиентов, да и самим работать. Но в целом услуга оказана. Но такая оценка именно мастеру. Больше к ней не пойду. Во-первых она не надела перчатки для проведения маникюра. Во вторых во время процедуры смотрела телевизор и издергала мне руки. Педикюр сделала хорошо, и в перчатках. 
Правильная оценка: 3
zero_shot: ответ='4', оценка=4, invalid=0
few_shot: ответ='2', оценка=2, invalid=0
